#  검색 성능 향상을 위한 기법 - 쿼리 확장 (Query Expansion)

- **학습 목표:** 쿼리 확장(Query Expansion) 기법을 구현하고 성능 개선을 측정한다

--- 

## 환경 설정 및 준비

**필수 데이터 파일**
- `./chroma_db`: 벡터 스토어 데이터베이스 디렉토리
- `data/testset.xlsx`: 평가용 테스트 데이터셋

**주요 패키지**
- `ranx-k` - 한국어 최적화 검색 성능 평가 라이브러리 (Kiwi 형태소 분석기 기반)

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) 벡터스토어 로드`

In [4]:
# 벡터 저장소 로드 
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`(5) 벡터 검색기 생성`

In [5]:
# 기본 retriever 초기화
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 4}
)

query = "리비안의 사업 경쟁력은 어디서 나오나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**재정**

Rivian의 재무 성과는 상당한 수익 성장과 상당한 순손실로 특징지어집니다.

| 연도 | 수익 (백만 USD) | 순이익 (백만 USD) | 총 자산 (백만 USD) |
| ---- | --------------- | ----------------- | ------------------ |
| 2020 | 0               | -1,018            | 4,602              |
| 2021 | 55              | -4,688            | 22,294             |
| 2022 | 1,658           | -6,752            | 17,876             |
| 2023 | 4,434           | -5,432            | 16,778             |

**최대 주주**

2023년 12월 현재 최대 주주는 Amazon, T. Rowe Price Inte

---

## 쿼리 확장 (Query Expansion)

1. **Query Reformulation**
    - LLM을 사용하여 원래 질문을 다른 형태로 재작성하는 방식임
    - 동의어 확장, 질문 명확화, 관련 키워드 추가 등 다양한 방식으로 쿼리를 변형함
    - 검색의 다양성과 정확도를 향상시키는 특징이 있음

1. **Multi Query** 
    - Retriever에 지정된 LLM을 활용하여 원본 쿼리를 확장하는 방법임
    - 하나의 질문에 대해 다양한 관점과 표현으로 여러 개의 쿼리를 자동 생성함
    - LLM의 생성 능력을 활용하여 검색의 다양성과 포괄성을 향상시키는 특징이 있음

1. **Decomposition** 
    - 복잡한 질문을 여러 개의 단순한 하위 질문으로 분해하는 LEAST-TO-MOST PROMPTING 전략을 사용함
    - 각각의 하위 질문에 대해 개별적으로 검색을 수행하여 더 정확한 답변을 도출함
    - 복잡한 질문을 체계적으로 해결하면서 검색의 정확도를 높이는 특징이 있음

1. **Step-Back Prompting**
    - 주어진 구체적인 질문에서 한 걸음 물러나 더 일반적인 개념이나 배경을 먼저 검색함
    - 더 넓은 맥락에서 점차 구체적인 답변으로 좁혀가는 방식을 사용함
    - 복잡한 질문에 대해 더 포괄적이고 정확한 답변을 제공하는 특징이 있음

1. **HyDE (Hypothetical Document Embedding)**
    - 주어진 질문에 대해 가상의 이상적인 답변 문서를 LLM으로 생성함
    - 생성된 가상 문서를 임베딩하여 이를 기반으로 실제 문서를 검색하는 방식임
    - 질문의 맥락을 더 잘 반영한 검색이 가능한 특징이 있음

### 1) **Query Reformulation** 

- **Query Reformulation**은 **LLM**을 활용해 원본 질문을 다양한 형태로 재구성
- **동의어 확장**과 **키워드 추가**를 통해 검색 쿼리의 범위를 확장
- 모호한 질문을 **명확하게 구체화**하여 검색 정확도 향상
- 하나의 질문에 대해 **다양한 변형 쿼리**를 생성하여 검색 커버리지 확대

<center>
<img src="ref/query_rewrite.png" alt="rag" align="center" border="0"  width="800" height=auto>
</center>


[출처] https://arxiv.org/abs/2305.14283

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 쿼리 리포뮬레이션을 위한 프롬프트 템플릿 정의
reformulation_template = """다음 질문을 검색 성능을 향상시키기 위해 다시 작성해주세요:
[질문]
{question}

다음 방식으로 질문을 재작성하세요:
1. 동의어 추가
2. 더 구체적인 키워드 포함
3. 관련된 개념 확장

[재작성된 질문]
"""

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template(reformulation_template)

# LLM 모델 초기화
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

# 쿼리 리포뮬레이션 체인 생성
reformulation_chain = prompt | llm | StrOutputParser()

# 체인 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
reformulated_query = reformulation_chain.invoke({"question": query})

print(f"쿼리: {query}")
pprint(f"리포뮬레이션된 쿼리: \n{reformulated_query}")

쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
('리포뮬레이션된 쿼리: \n'
 '리비안(Rivian)의 사업 경쟁력은 어떤 요소들에서 비롯되며, 전기 픽업트럭 및 SUV 시장 내에서의 강점과 차별화 전략은 무엇인가요?')


In [7]:
# 리포뮬레이션된 쿼리로 검색
retrieved_docs = chroma_k_retriever.invoke(reformulated_query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**Volkswagen과의 파트너십 (2024)**

- 2024년 6월, Volkswagen Group은 전기 아키텍처 및 소프트웨어 기술 개발을 목표로 Rivian에 최대 50억 달러를 투자할 의향을 발표.

**차량**

- **R1T:** 4개의 전기 모터가 장착된 픽업 트럭. 배터리 크기는 105 kWh에서 180 kWh까지 다양함.
- **R1S:** 첫 번째 Rivian 플랫폼의 스포츠 유틸리티 차량(SUV) 버전.
- **Electric Delivery Van (EDV):** 상업용 전기 밴으로, 주로 Amazon용으로 설계되어 사용됨.
- **R2:** 더 작고 저렴한 SUV로, 새로운 플랫폼에서 2026년 초에 출시될 예정.
- **R3:** 출시 예정인 전기 소형 SUV. [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- R1T 배송은 2021년 9월

In [8]:
# Runnable 객체로 변환하여 검색기 생성 (LCEL)
reformulation_retriever = reformulation_chain | chroma_k_retriever

# 쿼리 리포뮬레이션 검색기 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
retrieved_docs = reformulation_retriever.invoke({"question": query})

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**시설**

- **Irvine, California:** 차량 엔지니어링 및 설계에 중점을 둔 본사.
- **Normal, Illinois:** 차량 부품을 생산하고 조립을 수행하는 제조 공장.
- **Plymouth, Michigan:** 차량 엔지니어링, 프로토타입 제작, 공급망 및 회계에 중점을 둡니다.
- **Palo Alto, California:** 소프트웨어 개발 및 엔지니어링에 중점을 둡니다.
- Carson, California 및 Woking, England에 추가 사무실이 있습니다.
- 애틀랜타 동쪽에 있는 새로운 50억 달러 규모의 배터리 및 조립 공장은 보류 중입니다.

**재정**

Rivian의 재무 성과는 상당한 수익 성장과 상당한 순손실로 특징지어집니다. [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**Volkswagen과의 파트너십 

---
### **[실습 1]**

- 쿼리 리포뮬레이션 체인을 개선합니다. 
- langfuse Tracing에서 로그를 확인하고 쿼리 변환 과정을 이해합니다. 

**힌트**
1. `reformulation_template`의 프롬프트를 수정하여 더 효과적인 쿼리 변환을 시도해보세요
2. `langfuse_handler`를 체인에 추가하여 실행 과정을 추적하세요
   ```python
   reformulated_query = reformulation_chain.invoke(
       {"question": query},
       config={"callbacks": [langfuse_handler]}
   )
   ```
3. Langfuse 대시보드에서 생성된 쿼리와 검색 결과를 비교 분석하세요

In [12]:
# 여기에 코드를 작성하세요.

reformulation_template = """다음 질문을 검색 성능을 향상시키기 위해 다시 작성해주세요:
[질문]
{question}

다음 방식으로 질문을 재작성하세요:
1. 동의어 추가
2. 더 구체적인 키워드 포함
3. 관련된 개념 확장
4. 질문에 대한 카테고리를 지정

[재작성된 질문]
""""자료 원본"

prompt = ChatPromptTemplate.from_template(reformulation_template)

llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

reformulation_chain = prompt | llm | StrOutputParser()

query = "리비안의 사업 경쟁력은 어디서 나오나요?"
reformulated_query = reformulation_chain.invoke(
       {"question": query},
       config={"callbacks": [langfuse_handler]}
   )

print(f"쿼리: {query}")
pprint(f"리포뮬레이션된 쿼리: \n{reformulated_query}")

retrieved_docs = chroma_k_retriever.invoke(reformulated_query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)


쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
('리포뮬레이션된 쿼리: \n'
 '리비안(Rivian)의 사업 경쟁력은 어떤 요소들에서 비롯되며, 전기차 산업 내에서의 강점과 차별화 전략은 무엇인가요? (산업 분석 / '
 '전기차 제조 / 경쟁력 요인)')
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**시설**

- **Irvine, California:** 차량 엔지니어링 및 설계에 중점을 둔 본사.
- **Normal, Illinois:** 차량 부품을 생산하고 조립을 수행하는 제조 공장.
- **Plymouth, Michigan:** 차량 엔지니어링, 프로토타입 제작, 공급망 및 회계에 중점을 둡니다.
- **Palo Alto, California:** 소프트웨어 개발 및 엔지니어링에 중점을 둡니다.
- Carson, California 및 Woking, England에 추가 사무실이 있습니다.
- 애틀랜타 동쪽에 있는 새로운 50억 달러 규모의 배터리 및 조립 공장은 보류 중입니다.

**재정**

Rivian의 재무 

### 2) **Multi Query** 

- **Multi Query**는 **Retriever의 LLM**을 사용해 단일 질문을 다수의 쿼리로 확장
- 원본 질문에 대해 **다양한 관점**과 **표현 방식**으로 쿼리 자동 생성
- **LLM의 생성 능력**을 활용해 검색 범위를 자연스럽게 확장
- 검색의 **다양성**과 **포괄성**이 향상되어 관련 문서 검색 확률 증가

`(1) MultiQueryRetriever 활용`

- https://python.langchain.com/docs/how_to/MultiQueryRetriever/

In [13]:
# 멀티 쿼리 생성
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-nano',
    temperature=0.7,
)

# 기본 retriever를 이용한 멀티 쿼리 생성 
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=chroma_k_retriever, llm=llm
)

query = "리비안의 사업 경쟁력은 어디서 나오나요?"
retrieved_docs = multi_query_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:** [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**시설**

- **Irvine, California:** 차량 엔지니어링 및 설계에 중점을 둔 본사.
- **Normal, Illinois:** 차량 부품을 생산하고 조립을 수행하는 제조 공장.
- **Plymouth, Michigan:** 차량 엔지니어링, 프로토타입 제작, 공급망 및 회계에 중점을 둡니다.
- **Palo Alto, California:** 소프트웨어 개발 및 엔지니어링에 중점을 둡니다.
- Carson, California 및 Woking, England에 추가 사무실이 있습니다.
- 애틀랜타 동쪽에 있는 새로운 50억 달러 규모의 배터리 

`(2) Custom Prompt 활용`

In [14]:
from typing import List

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


# 모델 초기화
llm = ChatOpenAI(model="gpt-4.1-mini")

# 출력 파서: LLM 결과를 질문 리스트로 변환
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        """Split the text into lines and remove empty lines."""
        return [line.strip() for line in text.strip().split("\n") if line.strip()]
    

# 쿼리 생성 프롬프트
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""Generate three different versions of the given user question to retrieve relevant documents from a vector database. The goal is to reframe the question from various perspectives to overcome limitations of distance-based similarity search.

    The generated questions should have the following characteristics:
    1. Maintain the core intent of the original question but use different expressions or viewpoints.
    2. Include synonyms or related concepts where possible.
    3. Slightly broaden or narrow the scope of the question to potentially include diverse relevant information.

    Write each question on a new line and include only the questions.

    [Original question]
    {question}
    
    [Alternative questions]
    """,
)

# 멀티쿼리 체인 구성
multiquery_chain = QUERY_PROMPT | llm | LineListOutputParser()

# 테스트 쿼리 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
result = multiquery_chain.invoke({"question": query})

print("생성된 대안 질문들:")
for i, q in enumerate(result, 1):
    print(f"{i}. {q}")

생성된 대안 질문들:
1. 리비안이 시장에서 경쟁 우위를 확보하는 주요 요소는 무엇인가요?
2. 리비안의 강점과 차별화된 사업 전략은 어떤 부분에서 기인하나요?
3. 리비안이 경쟁 기업들과 비교해 가지는 사업적 경쟁력의 근원은 무엇인가요?


In [15]:
# 다중 쿼리 검색기 생성
multi_query_custom_retriever = MultiQueryRetriever(
    retriever=chroma_k_retriever, # 기본 retriever
    llm_chain=multiquery_chain,   # 멀티쿼리 체인
    parser_key="lines"            # "lines": 출력 파서의 키
)  

retrieved_docs = multi_query_custom_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**소송**

- 2020년 7월, Tesla는 Rivian이 독점 정보를 훔치고 직원을 빼갔다고 주장하며 소송을 제기했습니다.
- 2021년 3월, Illinois Automobile Dealers Association은 Rivian과 Lucid Motors가 소비자에게 직접 판매했다는 이유로 소송을 제기했습니다.
- 2021년 11월, 전 VP Laura Schwab은 차별 혐의로 소송을 제기하고 차량 가격 책정 및 안전 표준에 대한 우려를 제기했습니다. [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**재정**

Rivian의 재무 성과는 상당한 수익 성장과 상당한 순손실로 특징지어집니다.

| 연도 | 수익 (백만 USD) | 순이익 (백만 USD) | 총 자산 (백만 USD) |
| ---- | --------------- | ----------------- |

---
### **[실습 2]**

- 멀티쿼리 체인의 구조를 분석하고 개선합니다.  
- langfuse Tracing에서 로그를 확인하고 쿼리 변환 과정을 이해합니다. 

**힌트**
1. `QUERY_PROMPT` 템플릿을 수정하여 더 다양한 관점의 질문을 생성하도록 개선해보세요
2. 생성되는 대안 질문의 개수를 조정해보세요 (현재 3개)
3. `multi_query_custom_retriever`에 langfuse_handler를 추가하여 추적하세요
   ```python
   retrieved_docs = multi_query_custom_retriever.invoke(
       query,
       config={"callbacks": [langfuse_handler]}
   )
   ```
4. Langfuse에서 각 대안 질문별 검색 결과를 확인하고 중복 제거 과정을 분석하세요

In [16]:
# 여기에 코드를 작성하세요.

### 3) **Decomposition** 

- **단계별 분해 전략**을 통해 복잡한 질문을 작은 단위로 나누어 처리함
- 각 하위 질문마다 **독립적인 검색 프로세스**를 진행하여 정확도를 향상시킴
- **LEAST-TO-MOST PROMPTING**을 활용하여 체계적인 문제 해결 방식을 구현함
- 복잡한 문제를 단순화하여 검색 효율성을 극대화하는 방법론

<center>
<img src="ref/query_decomposition.png" alt="rag" align="center" border="0"  width="800" height=auto>
</center>


[출처] https://arxiv.org/pdf/2205.10625

In [17]:
from langchain_core.prompts import PromptTemplate
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to decompose the given input question into multiple sub-questions. 
    The goal is to break down the input into a set of sub-problems/sub-questions that can be answered independently.

    Follow these guidelines to generate the sub-questions:
    1. Cover various aspects related to the core topic of the original question.
    2. Each sub-question should be specific, clear, and answerable independently.
    3. Ensure that the sub-questions collectively address all important aspects of the original question.
    4. Consider temporal aspects (past, present, future) where applicable.
    5. Formulate the questions in a direct and concise manner.

    [Input question] 
    {question}

    [Sub-questions (5)]
    """,
)

# 쿼리 생성 체인
decomposition_chain = QUERY_PROMPT | llm | LineListOutputParser()

# 테스트 쿼리 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
result = decomposition_chain.invoke({"question": query})

print("생성된 서브 질문들:")
for i, q in enumerate(result, 1):
    print(f"{i}. {q}")

생성된 서브 질문들:
1. 1. 리비안의 주요 사업 영역은 무엇인가요?
2. 2. 리비안이 경쟁사와 차별화하는 기술적 강점은 무엇인가요?
3. 3. 리비안의 제품 라인업과 시장 반응은 어떠한가요?
4. 4. 리비안의 생산 능력과 공급망 현황은 어떻게 되나요?
5. 5. 리비안이 미래 성장 동력으로 삼고 있는 전략은 무엇인가요?


In [18]:
# 다중 쿼리 검색기 생성
multi_query_decompostion_retriever = MultiQueryRetriever(
    retriever=chroma_k_retriever,    # 기본 retriever
    llm_chain=decomposition_chain,   # 서브 질문 생성 체인
    parser_key="lines"               # "lines": 출력 파서의 키
)  

retrieved_docs = multi_query_decompostion_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:** [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**시설**

- **Irvine, California:** 차량 엔지니어링 및 설계에 중점을 둔 본사.
- **Normal, Illinois:** 차량 부품을 생산하고 조립을 수행하는 제조 공장.
- **Plymouth, Michigan:** 차량 엔지니어링, 프로토타입 제작, 공급망 및 회계에 중점을 둡니다.
- **Palo Alto, California:** 소프트웨어 개발 및 엔지니어링에 중점을 둡니다.
- Carson, California 및 Woking, England에 추가 사무실이 있습니다.
- 애틀랜타 동쪽에 있는 새로운 50억 달러 규모의 배터리 

---
### **[실습 3]**

- 쿼리 분해 체인의 구조를 분석하고 개선합니다.  
- langfuse Tracing에서 로그를 확인하고 쿼리 변환 과정을 이해합니다. 

**힌트**
1. `decomposition_chain`의 프롬프트를 수정하여 서브 질문 생성 전략을 개선해보세요
2. 서브 질문의 개수를 조정해보세요 (현재 5개)
3. 각 서브 질문이 독립적으로 답변 가능한지 확인하세요
4. langfuse_handler를 추가하여 분해 과정을 추적하세요
   ```python
   retrieved_docs = multi_query_decompostion_retriever.invoke(
       query,
       config={"callbacks": [langfuse_handler]}
   )
   ```
5. Langfuse에서 각 서브 질문별 검색 결과의 품질을 비교 분석하세요

In [19]:
# 여기에 코드를 작성하세요.

### 4) **Step-Back Prompting**

- **단계적 후퇴 방식**을 통해 구체적 질문을 일반적 맥락에서 접근함
- **맥락 기반 검색**으로 넓은 관점에서 구체적 답변으로 좁혀나감
- **포괄적 접근법**을 활용하여 복잡한 질문에 대한 이해도를 높임
- 일반적 맥락에서 시작하여 구체적 해답을 찾아가는 체계적 접근 방식

[출처] https://arxiv.org/pdf/2310.06117

`(1) Step-Back 질문 생성`

In [20]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Few Shot 예제 - (구체적 질문, 포괄적 질문) 쌍
examples = [
    {
        "input": "애플의 M1 칩 개발이 기업 가치에 미친 영향은?",
        "output": "기업의 핵심 기술 내재화가 경쟁우위에 미치는 영향은 무엇인가?",
    },
    {
        "input": "아마존의 AWS가 수익성에 기여하는 방식은?",
        "output": "기업의 새로운 사업 영역 확장이 수익 구조에 미치는 영향은 무엇인가?",
    },
    {
        "input": "토요타의 하이브리드 기술 전략의 핵심은?",
        "output": "자동차 산업에서 친환경 기술 혁신이 기업 성장에 미치는 영향은 무엇인가?",
    }
]

# 프롬프트 템플릿 초기화
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Step-Back 생성을 위한 프롬프트
step_back_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """당신은 기업 분석 전문가입니다. 특정 기업에 대한 구체적인 질문을 해당 산업이나 비즈니스 전반의 일반적인 관점에서 
                재해석하는 것이 임무입니다. 산업 동향, 경쟁 구도, 기술 혁신, 사업 모델 등의 관점에서 더 포괄적인 질문으로 
                바꾸어 주세요. 다음은 예시입니다:"""
            ),
            few_shot_prompt,
            ("user", "{question}"),
        ])

# Step-Back 체인 생성
step_back_chain = step_back_prompt | llm | StrOutputParser()

# Step-Back 질문 생성
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
step_back_question = step_back_chain.invoke({"question": query})

print(f"쿼리: {query}")
print(f"Step-Back 질문: {step_back_question}")

쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
Step-Back 질문: 전기차 신생 기업이 기존 자동차 산업 경쟁구도에서 차별화된 경쟁력을 확보하는 전략은 무엇인가?


In [21]:
# Step-Back 검색기 생성
step_back_retriever = step_back_chain | chroma_k_retriever

# Step-Back 검색 실행
retrieved_docs = step_back_retriever.invoke({"question": query})

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**소송**

- 2020년 7월, Tesla는 Rivian이 독점 정보를 훔치고 직원을 빼갔다고 주장하며 소송을 제기했습니다.
- 2021년 3월, Illinois Automobile Dealers Association은 Rivian과 Lucid Motors가 소비자에게 직접 판매했다는 이유로 소송을 제기했습니다.
- 2021년 11월, 전 VP Laura Schwab은 차별 혐의로 소송을 제기하고 차량 가격 책정 및 안전 표준에 대한 우려를 제기했습니다. [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**재정**

Rivian의 재무 성과는 상당한 수익 성장과 상당한 순손실로 특징지어집니다.

| 연도 | 수익 (백만 USD) | 순이익 (백만 USD) | 총 자산 (백만 USD) |
| ---- | --------------- | ----------------- | ------------------ |
| 2020 | 0               | -1,018            | 4,602              |
| 2021 | 55              | -4,688            | 22,294             |
| 2022 | 1,658           | -6,752            | 17,876             |
| 2023 | 4,434           | -5,432            | 16,778             |

**최대 주주**

2023년 12월 현재 최대 주주는 Amazon, T. Rowe Price International, The Vanguard Group, BlackRock 및 Fidelity Investments였습니다.

**협력**

`(2) 최종 답변 생성`

In [22]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate


# 프롬프트 템플릿 초기화
response_prompt = ChatPromptTemplate.from_template(
            """당신은 전문가입니다. 다음 컨텍스트와 질문을 바탕으로 포괄적인 답변을 제공해주세요.

            검색 결과 (직접 컨텍스트):
            {direct_context}
            
            상위 개념 (일반 컨텍스트):
            {step_back_context}
            
            원래 질문: {question}
            
            답변:"""
        )

# 문서 포맷팅 함수
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])


# 답변 생성 체인
answer_chain = (
            {
                "direct_context": chroma_k_retriever,
                "step_back_context": step_back_retriever,
                "question": RunnablePassthrough(),
            }
            | response_prompt
            | llm
            | StrOutputParser()
        )

# 답변 생성
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
answer = answer_chain.invoke(query)

print(f"쿼리: {query}")
print(f"답변: {answer}")

쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
답변: 리비안(Rivian)의 사업 경쟁력은 전기차 시장에서 독특한 위치와 전략, 기술력, 파트너십, 그리고 핵심 역량에서 비롯됩니다. 아래 주요 요소를 중심으로 설명드리겠습니다.

---

### 1. **전기차 및 야외 레크리에이션에 특화된 제품 전략**
- 리비안은 순수 전기 자동차(EV)와 배터리 기술을 중심으로 하는 회사로, 특히 전기 픽업트럭(R1T)과 전기 SUV(R1S)를 개발해 야외 레크리에이션과 모험을 즐기는 고객층을 공략하고 있습니다.
- 2021년 9월, 리비안은 완전 전기 픽업트럭을 소비자 시장에 출시한 최초의 회사 중 하나로서 이 부문에서 선구자적 위치를 확보했습니다.
- 이처럼 틈새 시장에 집중함으로써 차별화된 브랜드 이미지를 구축하고, 특화된 고객 니즈를 충족시키는 점이 강점입니다.

### 2. **기술 및 제품 경쟁력**
- 자체 배터리 및 전기 파워트레인 개발 역량을 보유하고 있으며, 전기차 전용 플랫폼을 통해 효율적이고 성능이 뛰어난 차량을 생산합니다.
- 다수의 엔지니어링 및 소프트웨어 개발 시설(예: 캘리포니아 어바인 본사, 미시간 플리머스, 캘리포니아 팔로알토 등)을 통해 차량 설계, 소프트웨어, 프로토타입 제작, 공급망 관리 등에서 기술 경쟁력을 강화하고 있습니다.

### 3. **확장 중인 생산 및 인프라**
- 현재 일리노이 주 노멀 공장에서 차량 생산과 부품 조립을 수행 중이며, 미국 동부 애틀랜타 인근에 50억 달러 규모의 대형 배터리 및 조립 공장을 계획해 향후 생산 능력 확장에 대비하고 있습니다.
- 이러한 생산 기반 확대 전략은 향후 수요 증가에 대응하고 공급망 안정화에 유리하게 작용할 것입니다.

### 4. **강력한 투자자 및 파트너십 네트워크**
- 2023년 기준 최대 주주로 Amazon, T. Rowe Price, Vanguard, BlackRock, Fidelity Investments 등 글로벌 주요 기관투자자를 확보해 탄탄한 자본력을 갖추고

---
### **[실습 4]**

- Step back 체인의 구조를 분석하고 개선합니다.  
- langfuse Tracing에서 로그를 확인하고 쿼리 변환 과정을 이해합니다. 

**힌트**
1. Few-shot 예제를 추가하거나 수정하여 Step-Back 질문의 품질을 개선해보세요
2. `step_back_prompt`의 시스템 메시지를 수정하여 더 나은 일반화를 유도하세요
3. `answer_chain`에 langfuse_handler를 추가하여 전체 과정을 추적하세요
   ```python
   answer = answer_chain.invoke(
       query,
       config={"callbacks": [langfuse_handler]}
   )
   ```
4. Langfuse에서 일반 컨텍스트와 Step-Back 컨텍스트의 차이를 비교하세요
5. 두 가지 컨텍스트를 결합한 답변의 품질을 평가하세요

In [23]:
# 여기에 코드를 작성하세요.

### 5) **HyDE** (Hypothetical Document Embedding)

- **가상 문서 생성**을 통해 주어진 질문에 대해 가상의 이상적인 답변 문서를 LLM으로 생성함
- 생성된 문서의 **임베딩 기반 검색**으로 실제 문서와 매칭을 수행함
- **맥락 기반 검색 방식**으로 질문의 의도를 더 정확하게 반영함

<center>
<img src="ref/query_HyDE.png" alt="rag" align="center" border="0"  width="1000" height=auto>
</center>

[출처] https://arxiv.org/abs/2212.10496

`(1) 가상 문서 생성`

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# HyDE를 위한 프롬프트 템플릿 생성
template = """주어진 질문에 대한 이상적인 문서 내용을 생성해주세요.
문서는 학술적이고 전문적인 톤으로 작성되어야 합니다.

질문: {question}

문서 내용:"""

hyde_prompt = ChatPromptTemplate.from_template(template)

# LLM 모델 초기화
hyde_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# 문서 생성 체인 생성
hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()

# 문서 생성 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
hypothetical_doc = hyde_chain.invoke({"question": query})

print(f"쿼리: {query}")
print(f"문서 내용: {hypothetical_doc}")

쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
문서 내용: 리비안(Rivian)의 사업 경쟁력은 여러 핵심 요소에서 기인하며, 이들은 회사의 전략적 방향성과 기술적 우위에 기반을 두고 있습니다. 본 문서에서는 리비안의 경쟁력을 구성하는 주요 요인들을 학술적 관점에서 분석하고자 한다.

1. 기술 혁신과 제품 차별화
리비안은 전기차(EV) 시장에서 독자적인 기술력과 차별화된 제품 포트폴리오를 구축하고 있다. 특히, 고성능 전기 픽업트럭인 R1T와 전기 SUV인 R1S는 뛰어난 주행 성능, 내구성, 그리고 첨단 안전·편의 사양을 갖추고 있어 시장 내 경쟁 우위를 확보하고 있다. 이러한 제품들은 배터리 기술, 전기 구동 시스템, 그리고 첨단 운전자 지원 시스템(ADAS) 분야에서의 지속적인 연구개발(R&D)을 통해 경쟁력을 강화하였다.

2. 첨단 기술과 인프라 구축
리비안은 배터리 기술, 전기 구동장치, 그리고 차량 내 인포테인먼트 시스템 등에서 첨단 기술을 도입하고 있으며, 이를 통해 제품의 성능과 안전성을 높이고 있다. 또한, 자율주행 및 연결성 기술 개발에 적극 투자하여 미래 모빌리티 시장에서의 경쟁력을 확보하고 있다. 더불어, 충전 인프라와 관련된 전략적 파트너십을 통해 고객의 편의성을 증대시키고 있으며, 이는 시장 확대에 중요한 역할을 하고 있다.

3. 지속가능성 및 친환경 전략
리비안은 친환경적이고 지속가능한 모빌리티 솔루션 제공을 기업 전략의 핵심으로 삼고 있다. 재생 가능 에너지 활용, 친환경 소재 사용, 그리고 탄소 배출 저감 목표를 통해 브랜드 이미지를 강화하고 있으며, 이는 환경에 민감한 소비자층을 공략하는 데 유리하게 작용한다.

4. 시장 포지셔닝과 고객 중심 전략
리비안은 주로 프리미엄 시장과 오프로드, 레저용 차량 시장을 타겟으로 하여 차별화된 브랜드 이미지를 구축하였다. 고객의 니즈에 부합하는 맞춤형 서비스와 높은 품질의 고객 지원 시스템을 운영함으로써 고객 충성도를 높이고 있다. 또한, 초기 시장 진입 시 강력한 브랜드 스토리와 

`(2) 유사 문서 검색`

In [25]:
# 가상 문서를 기반으로 실제 문서 검색
    
retrieved_docs = chroma_k_retriever.invoke(hypothetical_doc)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**시설**

- **Irvine, California:** 차량 엔지니어링 및 설계에 중점을 둔 본사.
- **Normal, Illinois:** 차량 부품을 생산하고 조립을 수행하는 제조 공장.
- **Plymouth, Michigan:** 차량 엔지니어링, 프로토타입 제작, 공급망 및 회계에 중점을 둡니다.
- **Palo Alto, California:** 소프트웨어 개발 및 엔지니어링에 중점을 둡니다.
- Carson, California 및 Woking, England에 추가 사무실이 있습니다.
- 애틀랜타 동쪽에 있는 새로운 50억 달러 규모의 배터리 및 조립 공장은 보류 중입니다.

**재정**

Rivian의 재무 성과는 상당한 수익 성장과 상당한 순손실로 특징지어집니다. [출처: data\Rivian_KR.md]
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
**재정**

Rivian의 재무 성

`(3) 최종 답변 생성`

In [26]:
# 최종 RAG를 위한 프롬프트 템플릿 생성
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요:

컨텍스트:
{context}

질문: {question}

답변:"""

rag_prompt =  ChatPromptTemplate.from_template(template)

# RAG 체인 생성
rag_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
rag_chain = rag_prompt | rag_llm | StrOutputParser()
    
# RAG 실행
query = "리비안의 사업 경쟁력은 어디서 나오나요?"
context = format_docs(retrieved_docs)

answer = rag_chain.invoke({"context": context, "question": query})

print(f"쿼리: {query}")
print(f"답변: {answer}")

쿼리: 리비안의 사업 경쟁력은 어디서 나오나요?
답변: 리비안의 사업 경쟁력은 다음과 같은 요소들에서 비롯됩니다:

1. **전기차 및 배터리 기술력:** 리비안은 전기 자동차와 배터리 개발에 집중하여 친환경 차량 시장에서 경쟁력을 갖추고 있습니다. 특히, 차량 엔지니어링, 설계, 소프트웨어 개발 등 다양한 분야의 전문 인력을 보유하고 있어 기술적 우위를 확보하고 있습니다.

2. **전문화된 시설과 글로벌 네트워크:** 미국 캘리포니아, 일리노이, 미시간 등 여러 지역에 위치한 연구개발, 제조, 공급망 관련 시설을 통해 제품 품질과 생산 효율성을 높이고 있습니다. 또한, 영국 Woking과 애틀랜타의 배터리 및 조립 공장 건설 예정으로 글로벌 생산 역량을 강화하고 있습니다.

3. **전략적 파트너십과 협력:** 유명 인사 및 기관과의 파트너십(예: Alex Honnold, Ewan McGregor, Casa Pueblo 등)을 통해 브랜드 인지도와 시장 확장에 기여하고 있으며, 주요 투자자인 Amazon, T. Rowe Price, BlackRock 등과의 협력을 통해 재무적 안정성과 성장 기반을 마련하고 있습니다.

4. **시장 포지셔닝과 서비스 제공:** 북미 시장에 집중하며, 전기차 충전, 자동차 보험 등 다양한 서비스를 제공함으로써 고객 충성도와 시장 점유율을 높이고 있습니다.

5. **혁신과 야외 레크리에이션 시장 공략:** 자동차 기술뿐만 아니라 야외 레크리에이션 분야에 특화된 제품과 서비스를 통해 차별화된 경쟁력을 갖추고 있습니다.

이러한 기술력, 시설, 전략적 파트너십, 시장 집중력 등을 바탕으로 리비안은 전기차 시장 내에서 차별화된 경쟁력을 갖추고 있다고 볼 수 있습니다.


`(4) HyDE 체인 종합`

In [28]:
# Step 1. 가상 문서 생성
query = "테슬라의 경영진을 분석해주세요."

hypothetical_doc = hyde_chain.invoke({"question": query})

# Step 2. 유사 문서 검색
retrieved_docs = chroma_k_retriever.invoke(hypothetical_doc)

# Step 3. 최종 답변 생성
final_answer = rag_chain.invoke(
    {
        "context": format_docs(retrieved_docs), 
        "question": query
    }
)

print(f"쿼리: {query}")
print(f"답변: {final_answer}")

쿼리: 테슬라의 경영진을 분석해주세요.
답변: 제공된 컨텍스트에는 테슬라의 경영진에 대한 구체적인 정보가 포함되어 있지 않습니다. 따라서, 이 자료를 바탕으로 테슬라의 경영진을 분석하는 것은 불가능합니다. 만약 테슬라의 경영진에 대한 상세한 정보를 원하신다면, 별도의 자료나 공식 발표 자료를 참고하시는 것이 좋습니다.


---
### **[실습 5]**

- HyDE 체인의 구조를 분석하고 개선합니다.  
- langfuse Tracing에서 로그를 확인하고 쿼리 변환 과정을 이해합니다. 

**힌트**
1. `hyde_prompt`의 템플릿을 수정하여 더 효과적인 가상 문서를 생성하도록 개선하세요
2. 가상 문서의 길이와 스타일을 조정해보세요 (학술적, 기술적, 비즈니스 등)
3. 각 단계별로 langfuse_handler를 추가하여 추적하세요
   ```python
   # Step 1: 가상 문서 생성 추적
   hypothetical_doc = hyde_chain.invoke(
       {"question": query},
       config={"callbacks": [langfuse_handler]}
   )
   
   # Step 3: 최종 답변 생성 추적
   final_answer = rag_chain.invoke(
       {"context": format_docs(retrieved_docs), "question": query},
       config={"callbacks": [langfuse_handler]}
   )
   ```
4. Langfuse에서 생성된 가상 문서와 실제 검색된 문서의 유사도를 비교하세요
5. HyDE 방식과 일반 검색 방식의 결과 차이를 분석하세요

In [ ]:
# 여기에 코드를 작성하세요.

---
### **[실습 6]**

- 쿼리 확장 기법 중에서 한 가지 기법을 선택합니다. 
- 테스트 데이터셋에 대한 검색 성능을 평가합니다. 

In [ ]:
# 기존에 생성해 둔 테스트셋 로드
import pandas as pd
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

In [ ]:
# 테스트 데이터셋의 특정 행에 있는 컨텍스트 데이터를 Document 객체 리스트로 변환
from langchain_core.documents import Document

context_docs = []
for i, row in df_qa_test.iterrows():
    row_docs = []
    for doc in eval(row['reference_contexts']):
        row_docs.append(Document(page_content=doc))

    context_docs.append(row_docs)


print(f"컨텍스트 문서: {len(context_docs)}개 문서")
print("="*200)
print(context_docs[0])

- ranx-k를 사용한 검색 성능 평가

In [ ]:
from ranx_k.evaluation import evaluate_with_ranx_similarity

# 평가 데이터 준비
questions = df_qa_test['user_input'].tolist()

reference_contexts = []
for contexts in df_qa_test['reference_contexts']:
    docs = [Document(page_content=ctx) for ctx in eval(contexts)]
    reference_contexts.append(docs)

# Multi Query Retriever 평가 (ranx-k 사용)
results_multiquery = evaluate_with_ranx_similarity(
    retriever=multi_query_custom_retriever,
    questions=questions,
    reference_contexts=reference_contexts,
    k=3,
    method='kiwi_rouge',
    similarity_threshold=0.8,
)

print("Multi Query Retriever 평가 결과 (ranx-k):")
print(f"  Hit Rate@3: {results_multiquery.get('hit_rate@3', 0):.3f}")
print(f"  MRR: {results_multiquery.get('mrr', 0):.3f}")
print(f"  MAP@3: {results_multiquery.get('map@3', 0):.3f}")
print(f"  NDCG@3: {results_multiquery.get('ndcg@3', 0):.3f}")

---

## 연습문제

다음 연습문제를 통해 쿼리 확장 기법에 대한 이해를 확인해 보세요.

### 문제 1: MultiQueryRetriever 생성

아래 코드의 빈칸을 채워 다중 쿼리 검색기를 생성하세요.

In [ ]:
from langchain_classic.retrievers import ____  # 힌트: MultiQueryRetriever

# 다중 쿼리 검색기 생성
multi_query_retriever = ____(
    retriever=____,  # 힌트: 기본 검색기
    llm=____,        # 힌트: LLM 모델 
)

# 검색 수행
query = "테슬라의 배터리 기술은 무엇인가요?"
results = multi_query_retriever.____(query)  # 힌트: 실행 메서드

print(f"검색 결과: {len(results)}개")

### 문제 2: HyDE 체인 구성

아래 코드의 빈칸을 채워 HyDE (Hypothetical Document Embedding) 체인을 구성하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# HyDE 프롬프트: 질문에 대한 가상의 답변 문서 생성
hyde_prompt = ChatPromptTemplate.from_template(
    """다음 질문에 대한 답변을 포함하는 짧은 문서를 작성하세요.
    
질문: {question}

문서:"""
)

# 가상 문서 생성 체인
hypothetical_doc_chain = hyde_prompt | ____ | ____  # 힌트: llm과 StrOutputParser()

# HyDE 검색 체인: 가상 문서로 검색
hyde_retriever_chain = hypothetical_doc_chain | ____  # 힌트: 검색기의 invoke를 람다로 연결

# 테스트
query = "테슬라의 배터리 기술"
# hypothetical_doc = hypothetical_doc_chain.invoke({"question": query})
# print(f"가상 문서: {hypothetical_doc[:200]}...")